# 06 · Mobility and shared-setting exposure

**Question:** How can an event log connect daily activity to a toy exposure model?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Have Astra audit the event schema, explain the effect of ventilation, and identify what is missing from a TB natural-history model.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## From places to events
160 synthetic agents occupy homes, workplaces and shared vehicles for one day.
Four fixed source agents contribute an arbitrary dose. Dose is not infection probability, TB incidence, or a calibrated clinical quantity.
This intentionally small model is not Starsim and does not reproduce Kopanyo findings.


In [ ]:
ventilation_multiplier=2.0
baseline=mobility(ventilation=1.)
scenario=mobility(ventilation=ventilation_multiplier)
sus=baseline['susceptible']
assert baseline['trajectory'].shape==(48,160)
assert np.array_equal(baseline['trajectory'],scenario['trajectory'])
assert np.allclose(scenario['dose'],baseline['dose']/ventilation_multiplier)
print('Shared-setting event rows:',len(baseline['events']))
print('Mean dose among non-source agents:',baseline['dose'][sus].mean(),'->',scenario['dose'][sus].mean())


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].imshow(baseline['trajectory'][:,:30].T,aspect='auto',origin='lower',extent=[0,24,0,30],cmap='tab20')
axes[0].set(xlabel='Hour',ylabel='Agent',title='First 30 activity histories (place IDs)')
axes[1].hist([baseline['dose'][sus],scenario['dose'][sus]],bins=15,label=['Baseline','More ventilation'])
axes[1].set(xlabel='Arbitrary dose',ylabel='Agents',title='Shared trajectories; changed exposure'); axes[1].legend()
plt.tight_layout(); plt.show()
export_json('06_events.json',dict(synthetic=True,events=baseline['events'],ventilation_multiplier=ventilation_multiplier))


## Teaching lab: Review an event contract
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 06_events.json and the arbitrary-dose assumptions. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Identify which fields establish shared occupancy and which information would still be needed for a disease model. Do not interpret dose as infection probability.

**Implementation brief — review the runnable extension below:**
> Compare ventilation multipliers 1, 2 and 4 using the same trajectories, then repeat with three fixed seeds. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** Within each seed, trajectories are identical across interventions and dose scales inversely with ventilation.

The extended investigation later in this notebook implements this line of work with maps and checks.
Use the brief to review the implementation or ask Codex for a further controlled variation.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extended investigation: Link activity histories to places, routes and contact layers

Attach synthetic place IDs to a building inventory and draw a vehicle route along the street grid. Compare ventilation 1×, 2× and 4× across three fixed seeds using common trajectories within each seed. Split events into household, workplace and vehicle layers. Vehicle positions are illustrative route locations; the original coarse half-hour activity schedule is not GPS tracking or a disease kernel.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Link activity histories to places, routes and contact layers', ['Building IDs + shared events', 'Homes / work / vehicle layers', 'Route map + seed comparison', 'Arbitrary dose ≠ disease'])
plt.show()


In [ ]:
ex_c=city(); ex_place_xy=ex_c['xy'][:48]; ex_nodes,ex_graph=network()
ex_route_nodes=list(range(11))+list(range(21,121,11)); ex_route=ex_nodes[ex_route_nodes]
assert all(any(v==b for v,w in ex_graph[a]) for a,b in zip(ex_route_nodes,ex_route_nodes[1:]))
ex_means=[]; ex_layers=[]
for seed in [7,17,27]:
    base=mobility(seed=seed); means=[]
    for ventilation in [1.,2.,4.]:
        run=mobility(seed=seed,ventilation=ventilation)
        assert np.array_equal(run['trajectory'],base['trajectory'])
        assert np.allclose(run['dose'],base['dose']/ventilation)
        means.append(float(run['dose'][run['susceptible']].mean()))
    ex_means.append(means)
    ex_layers.append([sum(e['people']*e['sources']*.02*.5 for e in base['events'] if (e['place']<40 if layer==0 else 40<=e['place']<48 if layer==1 else e['place']>=48)) for layer in range(3)])
ex_run=mobility(seed=7); ex_ids=ex_run['trajectory'][:,0]
ex_agent_xy=np.array([ex_place_xy[p] if p<48 else ex_route[(t*3)%len(ex_route)] for t,p in enumerate(ex_ids)])
fig,axes=plt.subplots(1,3,figsize=(14,4))
axes[0].scatter(*ex_place_xy[:40].T,s=12,label='Home places'); axes[0].scatter(*ex_place_xy[40:].T,marker='s',label='Work places')
axes[0].plot(*ex_route.T,c='#c68423',label='Illustrative vehicle route'); axes[0].scatter(*ex_agent_xy[[0,16,36]].T,c=['black','red','purple'],s=65)
map_axes(axes[0],'Places + street-constrained vehicle route'); axes[0].legend(fontsize=7)
for seed,values in zip([7,17,27],ex_means): axes[1].plot([1,2,4],values,marker='o',label=f'Seed {seed}')
axes[1].set(xlabel='Ventilation multiplier',ylabel='Mean arbitrary dose',title='Common trajectories, changed ventilation'); axes[1].legend(fontsize=8)
bottom=np.zeros(3)
for j,layer in enumerate(['Household','Workplace','Vehicle']):
    values=np.array(ex_layers)[:,j]; axes[2].bar(['7','17','27'],values,bottom=bottom,label=layer); bottom+=values
axes[2].set(xlabel='Seed',ylabel='Summed event dose (all agents)',title='Shared-setting layers'); axes[2].legend(fontsize=8)
fig.tight_layout(); extension_figure=fig
extension_evidence=dict(seeds=[7,17,27],ventilation=[1,2,4],mean_dose=ex_means,units='arbitrary dose',route_node_ids=ex_route_nodes)
extension_expected=dict(dose_ratio_4_to_1=.25,route_uses_street_edges=True)
extension_task='Return dose_ratio_4_to_1 and route_uses_street_edges. Infer the ratio from the fixed-trajectory experiment; grid nodes are row-major on an 11 by 11 grid and consecutive route nodes are listed.'


In [ ]:
extension_prompt,extension_reference=export_investigation('06',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 06_extension_prompt.json and optionally 06_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `06_extension_prompt.json` and `06_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('06_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
